In [ ]:
import numpy as np
import pandas as pd

In [ ]:
FILES = [
"I1.csv",
"I2.csv",
"I3.csv",
"I4.csv",
"I5.csv",
"I6.csv",
"I7.csv",
"I8.csv",
"I9.csv",
"I10.csv",
"I11.csv",
"I12.csv",
"I13.csv",
"I14.csv",
"I15.csv",
"I16.csv",
"I17.csv",
"I18.csv",
"I19.csv",
"I20.csv",
"I21.csv",
"I22.csv",
"I23.csv",
"I24.csv",
]
season_map = {
    f"I{i}": f"{2025 - i}/{2026 - i}" 
    for i in range(1, 25)
}
frames = []
for fp in FILES:
    df_i = pd.read_csv(fp, low_memory=False, on_bad_lines='skip')
    frames.append(df_i)

df_raw = pd.concat(frames, ignore_index=True)

df_raw.sample(n=20)

In [ ]:
need = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "MW"]
have = [c for c in need if c in df_raw.columns]
df = df_raw[have].copy()
df.sample(n=10)

In [ ]:
df["date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

def get_season(date):
    if pd.isna(date):
        return None
    year = date.year
    if date.month >= 8:  # August → season starts this year
        return f"{year}/{year+1}"
    else:  # Jan–July → season is last year
        return f"{year-1}/{year}"

df["season"] = df["date"].apply(get_season)

In [ ]:
df = df.rename(columns={"HomeTeam": "home_team", "AwayTeam": "away_team"})

df["hometeamgoals"] = df["FTHG"].astype("Int64") if "FTHG" in df.columns else pd.NA
df["awayteamgoals"] = df["FTAG"].astype("Int64") if "FTAG" in df.columns else pd.NA

# Fill NA values with 0 before comparison
df["hometeamgoals"] = df["hometeamgoals"].fillna(0)
df["awayteamgoals"] = df["awayteamgoals"].fillna(0)


df["hometeamresult"] = np.where(
    df["hometeamgoals"] > df["awayteamgoals"], "win",
    np.where(df["hometeamgoals"] < df["awayteamgoals"], "loss", "draw")
)

_points = {"win": (3, 0), "loss": (0, 3), "draw": (1, 1)}
df[["home_team_points", "away_team_points"]] = df["hometeamresult"].map(_points).apply(pd.Series)

if "MW" in df.columns:
    df = df.rename(columns={"MW": "week"})

In [ ]:
final_cols = ["season", "date"]

final_cols += [
    "home_team", "away_team",
    "hometeamgoals", "awayteamgoals",
    "hometeamresult", "home_team_points", "away_team_points",
]


final_df = df[final_cols].sort_values(
    ["season", "date"], 
    ascending=[False, False],   # season desc, date desc
    ignore_index=True
)

In [ ]:
final_df.to_csv("final_matches.csv", index=False)